<a href="https://colab.research.google.com/github/Ashu-Shukla-1309/supreme-goggles/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The Contract:

Unit of Analysis: One row equals one unique content item (content_hash_id) for a specific client (client_hash_id) aggregated at a point in time.

Time Window: A single mid-panel month (month=2026-03). We use the first 15 days as the "historical pre-decision window" and the latter 15 days to calculate the true outcome (label).

In [21]:
import os, duckdb
from google.colab import userdata

# 1. Re-fetch the token and re-authorize DuckDB
HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

fact_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# 2. Run the query
print("--- 1. ROW COUNT & DATE SPAN ---")
q1 = con.sql(f"SELECT MIN(report_date) as start_date, MAX(report_date) as end_date, COUNT(*) as total_rows FROM read_parquet('{fact_url}')").df()
print(q1)

--- 1. ROW COUNT & DATE SPAN ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (The 5 allowed): imp_prev, clk_prev, pos_prev, peak_imp_prev, best_pos_prev.

Label / Proxy: is_declining. A binary flag denoting if impressions in the second half of the month dropped by >20% compared to the first half.

Context: client_hash_id and content_hash_id. These strictly identify the unit but are not fed into the model.

Deliberately Excluded: imp_future (the raw future impressions count). We use it strictly to calculate the label and then immediately discard it. Passing future impressions directly into the training features is the ultimate data leak.

In [22]:
# Field categorization verified conceptually. Next cell executes the validation.
print("Fields categorized. Awaiting query validation.")

Fields categorized. Awaiting query validation.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The Five Features and 'Available When?' Justification:

imp_prev: Knowable at the decision moment because it aggregates historical Search Console impressions prior to the cutoff date.

clk_prev: Knowable at the decision moment because it aggregates historical click logs.

pos_prev: Knowable at the decision moment because ranking positions are recorded daily in the past.

peak_imp_prev: Knowable at the decision moment as it calculates the max daily impressions observed during the historical window.

best_pos_prev: Knowable at the decision moment as it retrieves the highest rank achieved before the decision date.

In [19]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# --- PROVE THREE FACTS ---
print("--- 1. ROW COUNT & DATE SPAN ---")
q1 = con.sql(f"SELECT MIN(report_date) as start_date, MAX(report_date) as end_date, COUNT(*) as total_rows FROM read_parquet('{fact_url}')").df()
print(q1)

print("\n--- 2. GRAIN VERIFICATION ---")
# Proving one row = one client + content + day in the raw data
q2 = con.sql(f"SELECT COUNT(*) as raw_rows, COUNT(DISTINCT client_hash_id || content_hash_id || report_date) as unique_grain FROM read_parquet('{fact_url}')").df()
print(q2)

print("\n--- 3. AVAILABILITY (IS TRUE filter) ---")
q3 = con.sql(f"SELECT COUNT(*) as valid_rows FROM read_parquet('{fact_url}') WHERE (gsc_impressions > 0) IS TRUE").df()
print(f"Rows surviving with >0 impressions: {q3['valid_rows'][0]:,}")

# We split the month: Days 1-15 are features, Days 16-31 are the future (label)
df = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) as imp_prev,
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) as clk_prev,
        AVG(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) as pos_prev,
        MAX(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) as peak_imp_prev,
        MIN(CASE WHEN report_date <= '2026-03-15' THEN gsc_avg_position END) as best_pos_prev,
        SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) as imp_future
    FROM read_parquet('{fact_url}')
    GROUP BY 1, 2
    HAVING imp_prev > 100
""").df()

# Create Label
df['is_declining'] = (df['imp_future'] < 0.8 * df['imp_prev']).astype(int)

# THE TRAP: Adding a leaky feature derived from the outcome window
df['LEAKY_future_ratio'] = df['imp_future'] / (df['imp_prev'] + 1)

features_honest = ['imp_prev', 'clk_prev', 'pos_prev', 'peak_imp_prev', 'best_pos_prev']
features_leaky = features_honest + ['LEAKY_future_ratio']

# Test both models
X_honest, X_leaky, y = df[features_honest].fillna(0), df[features_leaky].fillna(0), df['is_declining']
X_tr_h, X_te_h, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42)
X_tr_l, X_te_l, _, _ = train_test_split(X_leaky, y, test_size=0.25, random_state=42)

model_honest = RandomForestClassifier(max_depth=3, random_state=42).fit(X_tr_h, y_tr)
model_leaky = RandomForestClassifier(max_depth=3, random_state=42).fit(X_tr_l, y_tr)

print("\n--- THE LEAKAGE TRAP RESULTS ---")
print(f"Honest Model Precision: {precision_score(y_te, model_honest.predict(X_te_h)):.3f}")
print(f"Leaky Model Precision:  {precision_score(y_te, model_leaky.predict(X_te_l)):.3f} <- Perfect score illusion due to target leakage!")

# Remove the leak to maintain the honest data contract
df = df.drop(columns=['LEAKY_future_ratio', 'imp_future'])

--- 1. ROW COUNT & DATE SPAN ---
  start_date   end_date  total_rows
0 2026-03-01 2026-03-31     9841378

--- 2. GRAIN VERIFICATION ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   raw_rows  unique_grain
0   9841378       9841378

--- 3. AVAILABILITY (IS TRUE filter) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows surviving with >0 impressions: 3,611,061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- THE LEAKAGE TRAP RESULTS ---
Honest Model Precision: 0.620
Leaky Model Precision:  0.997 <- Perfect score illusion due to target leakage!


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitations of this slice:
The warehouse panel is unbalanced. By strictly isolating month=2026-03 as our window, any clients that were onboarded to the FlyRank platform after March 2026 are entirely excluded from this analysis. Furthermore, early history rows may lack robust GA4 engagement data depending on the client's integration date, meaning we cannot build universal features relying exclusively on cross-platform event metrics for this timeframe.

In [20]:
# Limitation noted conceptually.
print("Data limits documented.")

Data limits documented.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.